# KMeans Clustering

## Import data

This data comprises of the top three principal components obtained during PCA with a radial basis function kernel, which together contain ~95% of the total variance of the original dataset (as seen in the cumulative variance barplot from the PCA section).

In [24]:
import pandas as pd

df = pd.read_csv('../../data/kpca3d.csv')
df.head()

,PC1,PC2,PC3,All Time Rank,All Time Rank Bin,Track,Artist
0,0.150251,0.630721,-0.248971,1,1,MILLION DOLLAR BABY,Tommy Richman
1,-0.089829,-0.025684,-0.041813,2,1,Not Like Us,Kendrick Lamar
2,-0.021122,0.281381,-0.197222,3,1,i like the way you kiss me,Artemas
3,0.600836,0.611817,-0.103087,4,1,Flowers,Miley Cyrus
4,0.010005,0.400341,-0.205579,6,1,Lovin On Me,Jack Harlow


## Normalize data

In [26]:
from sklearn.preprocessing import StandardScaler

def normalize_data(df, quantitative_cols):
    """Use standard scaler to normalize quantitative data columns."""
    scaler = StandardScaler()
    return scaler.fit_transform(df[quantitative_cols])

In [28]:
df_scaled = normalize_data(df, ['PC1', 'PC2', 'PC3'])
df_scaled = pd.DataFrame(df_scaled)
df_scaled = df_scaled.rename(columns={0: 'PC1', 1: 'PC2', 2: 'PC3'})
df_scaled.describe()

,PC1,PC2,PC3
count,3.311000e+03,3.311000e+03,3.311000e+03
mean,-3.433610e-17,-2.575208e-17,8.584026e-18
std,1.000151e+00,1.000151e+00,1.000151e+00
min,-8.435623e-01,-1.562760e+00,-2.589789e+00
25%,-6.970511e-01,-5.067461e-01,-5.929945e-01
50%,-4.307679e-01,-9.299233e-02,4.344051e-02
75%,3.206849e-01,9.725160e-02,5.452515e-01
max,3.540588e+00,4.897467e+00,6.095500e+00


## Separate label and principal components

In [29]:
# extract label
label = df[['All Time Rank Bin']]

# extract three principal components
df_pca = df_scaled[['PC1', 'PC2', 'PC3']].values
pd.DataFrame(df_pca).head()

,0,1,2
0,0.562439,3.470437,-1.954463
1,-0.336259,-0.141324,-0.328238
2,-0.079065,1.548252,-1.548226
3,2.249132,3.366418,-0.809250
4,0.037453,2.202810,-1.613826


## Import modules

For silhouette scores, we use scikit-learn's silhouette_score:
1. silhouette_score: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html

For clustering, we use scikit-learn's KMeans:
1. KMeans: https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html

For plotting, we use Plotly's 3D scatterplot, seaborn, and Pyplot:
1. 3D: https://plotly.com/python/3d-scatter-plots/
2. seaborn: https://seaborn.pydata.org/
3. Pyplot: https://matplotlib.org/stable/tutorials/pyplot.html

In [21]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import plotly.graph_objects as go
import seaborn as sns
import numpy as np

## Use silhouette method to decide k-values

In [ ]:
def compute_silhouette_scores(X, k_values):
    """Compute and plot the silhouette score for a range of k-values starting at 2."""
    scores = []
    
    # use KMeans with 10 trials for a given k-value
    for k in k_values:
        clusterer = KMeans(n_clusters=k, n_init=10)
        labels = clusterer.fit_predict(X)
        score = silhouette_score(X, labels)
        scores.append(score)
    
    plt.figure(figsize=(10, 5))
    plt.plot(k_values, scores, marker='o', linestyle='-', color='mediumseagreen', alpha=0.7)
    plt.title('Silhouette Scores by Number of Clusters')
    plt.ylabel('Silhouette Score')
    plt.xlabel('Number of Clusters (k)')
    plt.grid(True)
    plt.show()

k_values = list(range(2, 8))
compute_silhouette_scores(df_pca, k_values)

## Visualize k=(2, 3, 4) clusters

In [ ]:
def visualize_clusters_3d(X, k_values, colors):
    """Create 3D scatterplots to visualize the clustering of the best performing k-values."""
    
    for k in k_values:
        # use KMeans with 10 trials for a given k-value
        clusterer = KMeans(n_clusters=k, n_init=10)
        labels = clusterer.fit_predict(X)
        cluster_centers = clusterer.cluster_centers_
        cluster_colors = [f"rgb({int(r*255)}, {int(g*255)}, {int(b*255)})" for r, g, b in colors]
        centroid_colors = [cluster_colors[i] for i in range(k)]

        fig_3d = go.Figure()

        # data points
        fig_3d.add_trace(go.Scatter3d(
            x=X[:, 0], y=X[:, 1], z=X[:, 2],
            mode='markers',
            marker=dict(size=2.5, color=[cluster_colors[i] for i in labels], opacity=0.5),
            name='Data Points'
        ))

        # centroids
        fig_3d.add_trace(go.Scatter3d(
            x=cluster_centers[:, 0], y=cluster_centers[:, 1], z=cluster_centers[:, 2],
            mode='markers+text',
            marker=dict(size=12, color='black', symbol='diamond'),
            text=[str(i+1) for i in range(k)],
            textposition='middle center',
            textfont=dict(color=centroid_colors, size=14, family='Arial Black'),
            name='Centroids'
        ))

        title = f'3D Clustered PCA Data (k={k})'
        fig_3d.update_layout(
            title=title,
            scene=dict(xaxis_title='PC1', yaxis_title='PC2', zaxis_title='PC3'),
            margin=dict(l=0, r=0, b=0, t=40),
            scene_camera=dict(
                eye=dict(x=1.3, y=-1.7, z=1.3),
                center=dict(x=0, y=0, z=0),
                up=dict(x=0, y=0, z=1),
            ),
            autosize=False, 
            width=1000, 
            height=800
        )

        # save Plotly's HTML files to display on a website
        # fig_3d.write_html(f"./kmeans-plots/{title.replace(' ', '')}.html")
        fig_3d.show()
            
# k-values with high silhouette scores
k_values = [2, 3, 4]

# set up colormap
colors = sns.color_palette('Set2')[:max(k_values)]

visualize_clusters_3d(df_pca, k_values, colors)